# 🚀 Build a shopping agent using Sequence Knowledge Distillation on Amazon SageMaker AI

**Sequence-level knowledge distillation (SeqKD)**: Supervised fine-tuning of `Qwen/Qwen3.5-4B` on a stronger teacher's
tool-use trajectories, using serverless `SFTTrainer` (SageMaker Python SDK v3).

**Task:** A shopper
asks for products (e.g. *"women's jeans in size eu:30 with a high waist, priced above 114 PHP (Philippine Currency)"*); the agent searches a
product catalog containing 2.7 million products with tools (`find_product`, `view_product_information`, `recommend_product`, `terminate`) and
must recommend product(s) meeting every constraint. Train a small student model to become the shopping agent instead of using a agentic frontier model. The product catalog, test dataset and the benchmark come from [ShoppingBench](https://github.com/yjwjy/ShoppingBench) ([paper](https://arxiv.org/abs/2508.04266)). 

**Data:** Teacher trajectories from [oro-ai/sn15-shoppingbench-sft-15k](https://huggingface.co/datasets/oro-ai/sn15-shoppingbench-sft-15k).

We will build this fine-tuned model using following steps:

1. Encode each trajectory step in the exact format the benchmark's agent loop sends and parses
2. Train with the serverless `SFTTrainer` (LoRA)
3. Retrieve the registered Model Package
4. Evaluate ASR and IFEval in the companion notebook `evaluate-tool-call-accuracy.ipynb`


## 00. Setup

In [ ]:
import os, json, boto3
from sagemaker.core.helper.session_helper import get_execution_role

REGION = boto3.Session().region_name or "us-west-2"
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"] = REGION

# MLflow metrics while the trainer runs
os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = f"https://mlflow.sagemaker.{REGION}.app.aws"

try:
    ROLE = get_execution_role()
    print("Execution role:", ROLE)
except Exception:
    ACCOUNT = boto3.client("sts").get_caller_identity()["Account"]
    ROLE = f"arn:aws:iam::{ACCOUNT}:role/<your-sagemaker-execution-role>"   # <- replace

BUCKET  = "<your-bucket>"                             # <- replace with your bucket
PREFIX  = "seqkd-shopping"
S3_ROOT = f"s3://{BUCKET}/{PREFIX}"
print(REGION, ROLE, S3_ROOT)

## 01. Data prep

### 01a. Load the teacher trajectories
We use teacher trajectories from [oro-ai/sn15-shoppingbench-sft-15k](https://huggingface.co/datasets/oro-ai/sn15-shoppingbench-sft-15k) that are created from a SN15 subnet arena where competing agents are incentivized to submit teacher trajectories and a reasoning judge picks the wining trajectory. The trajectories are further filtered for structural quality to create the teacher dataset.

In [ ]:
import json, random
from huggingface_hub import list_repo_files, hf_hub_download
from transformers import AutoTokenizer
HF_STUDENT = "Qwen/Qwen3.5-4B"
REPO = "oro-ai/sn15-shoppingbench-sft-15k"   # teacher trajectories
jsonls = [f for f in list_repo_files(REPO, repo_type="dataset") if f.endswith(".jsonl")]
sft_file = next(f for f in jsonls if "metadata" not in f.lower() and "eval" not in f.lower())
raw = [json.loads(l) for l in open(hf_hub_download(REPO, sft_file, repo_type="dataset")) if l.strip()]
raw = [r for r in raw if r.get("messages")]
print(len(raw), "trajectories")

# ShoppingBench code + test files, in the folder the eval notebook also uses (--code-only: no catalog or index)
import subprocess
from pathlib import Path
SHOPBENCH_WORK = Path.home() / "shopbench-eval"

subprocess.run(["bash", "scripts/setup_shopbench.sh", "--code-only", str(SHOPBENCH_WORK)], check=True)
SB_DIR = SHOPBENCH_WORK / "ShoppingBench"
test_rows = {t: [json.loads(l) for l in open(SB_DIR / "data" / f"synthesize_{t}_test.jsonl") if l.strip()]
             for t in ("product", "shop", "voucher")}

def traj_query(t):                      # a trajectory's task == its first user message
    return next((m["content"] for m in t["messages"] if m.get("role") == "user"), None)

train_queries = {traj_query(t) for t in raw}
for t, rows in test_rows.items():
    n = sum(r["query"] not in train_queries for r in rows)
    print(f"{t}: {n} / {len(rows)} test problems are held out (query not in the corpus)")

### 01b. Encode every step in the ShoppingBench harness format


| Message | Content |
|---|---|
| `user` | The agent instructions and tool list, and the dialogue history i.e. request + earlier tool calls and results |
| `assistant` (target) | The teacher's next action: `<tool_call>[{"name": …, "parameters": …}]</tool_call>` |



- The loss is computed by SFTTrainer on the assistant message only and not on the prompt.
- Tool arguments are normalized to what the harness accepts; steps with other tools are dropped; the split is by query.

In [ ]:
import subprocess, sys
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ujson", "pydantic", "portalocker", "openai",
                "duckduckgo_search", "googlesearch-python", "mcp", "colorama"], check=True)
sys.path.insert(0, str(SB_DIR / "src" / "agent"))
from run_rollout import get_system_prompt, get_user_prompt
from util.message import Message, generate_tool_call_id

SYSTEM_PROMPT = get_system_prompt({"system_prompt_file": str(SB_DIR / "src/agent/prompt/rollout.md"),
                                   "exclude_tools": ["web_search"]})   # web search: knowledge intent only
HARNESS_TOOLS = {"find_product", "view_product_information", "recommend_product", "terminate"}
tok = AutoTokenizer.from_pretrained(HF_STUDENT, trust_remote_code=True)

def harness_call(tc):
    """ORO tool call -> the argument shapes the harness tools accept."""
    f = tc.get("function", tc); name = f["name"]; a = f.get("arguments") or {}
    a = json.loads(a) if isinstance(a, str) else dict(a)
    if name == "find_product":
        if "q" not in a and "query" in a: a["q"] = a.pop("query")
        a.setdefault("page", 1)                                   # required by the harness's find_product
    elif name in ("view_product_information", "recommend_product"):
        v = a.pop("product_ids", a.pop("product_id", ""))
        a["product_ids"] = ",".join(map(str, v)) if isinstance(v, list) else str(v)   # comma-separated string
    elif name == "terminate":
        s = a.get("status", "success"); a["status"] = s if isinstance(s, str) else ("success" if s else "failure")
    return {"name": name, "parameters": a}

def to_rows(traj):
    msgs = [m for m in traj["messages"] if m.get("role") != "system"]
    query = next(m["content"] for m in msgs if m["role"] == "user")
    steps, i = [], 0
    while i < len(msgs):                     # each tool-calling assistant turn + the tool results after it
        m = msgs[i]; i += 1
        if m["role"] != "assistant" or not m.get("tool_calls"):
            continue                         # prose-only turns never occur in the harness loop
        calls = [harness_call(tc) for tc in m["tool_calls"]]
        if any(c["name"] not in HARNESS_TOOLS for c in calls):
            return None                      # uses a tool the ShoppingBench harness does not have
        results = []
        while i < len(msgs) and msgs[i]["role"] == "tool":
            r = msgs[i]["content"]
            try: r = json.loads(r)
            except Exception: pass
            results.append(r); i += 1
        steps.append((calls, results))
    if not steps:
        return None
    rows, history, prev = [], [], Message(user=query)
    for calls, results in steps:
        user_prompt = get_user_prompt(prev, history)               # appends prev to history, as the agent loop does
        # instructions at the top of the user message (see 01b above)
        rows.append({"messages": [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_prompt}"},
                                  {"role": "assistant", "content": f"<tool_call>{json.dumps(calls, ensure_ascii=False)}</tool_call>"}]})
        tcs = [dict(c, tool_call_id=generate_tool_call_id(c["name"], c["parameters"])) for c in calls]
        prev = Message(tool_call=tcs, obs=[{"tool_call_id": t["tool_call_id"], "results": r} for t, r in zip(tcs, results)])
        if any(c["name"] == "terminate" for c in calls):
            break                                                   # the harness loop stops here
    return rows

# Split by query (~30 trajectories share each query, so a step-level split would leak)
by_q = {}
for t in raw: by_q.setdefault(traj_query(t), []).append(t)
queries = sorted(by_q); random.seed(42); random.shuffle(queries)
val_q = set(queries[:max(1, len(queries) // 20)])                  # ~5% of queries held out
train, val, skipped = [], [], 0
for q, trajs in by_q.items():
    for t in trajs:
        rows = to_rows(t)
        if rows is None: skipped += 1; continue
        (val if q in val_q else train).extend(rows)
random.shuffle(train)
print(f"{len(raw)} trajectories ({skipped} skipped: non-harness tools or no tool calls) -> "
      f"train {len(train)} / val {len(val)} examples; {len(queries) - len(val_q)} / {len(val_q)} queries")

# Self-check: the harness parser must accept every target
target = lambda row: row["messages"][-1]["content"]
rate = sum(bool(Message.from_string("", target(r)).tool_call) for r in train[:2000]) / min(len(train), 2000)
print(f"harness parser accepts {rate:.1%} of targets"); assert rate > 0.99



Truncate the dataset examples to the maximum dataset length else the dataset gets truncated by SFTTrainer

In [ ]:
# Drop examples longer than dataset_max_len: the trainer truncates from the left
MAX_TOKENS = 16384                                                  # = dataset_max_len in step 02
n_tokens = lambda r: len(tok(tok.apply_chat_template(r["messages"], tokenize=False))["input_ids"])
for name, rows in (("train", train), ("val", val)):
    lens = [n_tokens(r) for r in rows]
    keep = [r for r, n in zip(rows, lens) if n <= MAX_TOKENS]
    s = sorted(lens)
    print(f"{name}: tokens per example median {s[len(s) // 2]}, p99 {s[int(len(s) * 0.99)]}, max {s[-1]}; "
          f"dropped {len(rows) - len(keep)} longer than {MAX_TOKENS}")
    rows[:] = keep
print("--- example (as the trainer renders it) ---\n" + tok.apply_chat_template(train[0]["messages"], tokenize=False))

# Write the files to local disk
TRAIN_FILE, VAL_FILE = "seqkd_train.jsonl", "seqkd_val.jsonl"
def write_jsonl(rows, p):
    with open(p, "w") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
write_jsonl(train, TRAIN_FILE); write_jsonl(val, VAL_FILE)

#### Upload and register the datasets

Use `DataSet.create` to create **registered dataset ARNs**

In [ ]:
from sagemaker.ai_registry.dataset import DataSet

s3 = boto3.client("s3", region_name=REGION)
s3.upload_file(TRAIN_FILE, BUCKET, f"{PREFIX}/data/{TRAIN_FILE}")
s3.upload_file(VAL_FILE,   BUCKET, f"{PREFIX}/data/{VAL_FILE}")
TRAIN_S3 = f"{S3_ROOT}/data/{TRAIN_FILE}"
VAL_S3   = f"{S3_ROOT}/data/{VAL_FILE}"

train_dataset = DataSet.create(name="seqkd-shopping-train", source=TRAIN_S3)
val_dataset   = DataSet.create(name="seqkd-shopping-val",   source=VAL_S3)
print("train:", train_dataset.arn)
print("val:  ", val_dataset.arn)

## 02. Train with the serverless `SFTTrainer`

Sequence Level Knowledge Distillation here is ordinary SFT on the teacher generated trajectories from the dataset.

In [ ]:
from sagemaker.train.sft_trainer import SFTTrainer
from sagemaker.train.common import TrainingType
from sagemaker.core.resources import ModelPackageGroup

MPG_NAME = "seqkd-qwen35-4b-shopping"
try:
    model_package_group = ModelPackageGroup.create(model_package_group_name=MPG_NAME)
except Exception as e:
    print("reusing existing group:", type(e).__name__)
    model_package_group = MPG_NAME          # an existing name or ARN is also accepted

# SFTTrainer.list_supported_models() lists the models serverless training supports

sft_trainer = SFTTrainer(
    model="huggingface-vlm-qwen3-5-4b",     # Qwen3.5-4B
    training_type=TrainingType.LORA,        # Use TrainingType.FULL for full fine-tuning
    model_package_group=model_package_group,
    mlflow_experiment_name="seqkd-shopping-exp",
    training_dataset=train_dataset.arn,
    validation_dataset=val_dataset.arn,
    s3_output_path=f"{S3_ROOT}/output/",
    accept_eula=True,                       #To accept licensing for the models
)

### Hyperparameters

Print the recipe defaults for the model and training type, then override the hyperparameters you want to adjust. Set them **after** creating the trainer (re-creating it resets them).

In [ ]:
from pprint import pprint
print("Default hyperparameters")
pprint(sft_trainer.hyperparameters.to_dict())

In [ ]:
sft_trainer.hyperparameters.dataset_max_len       = 16384
sft_trainer.hyperparameters.max_epochs            = 2
sft_trainer.hyperparameters.global_batch_size     = 64
sft_trainer.hyperparameters.learning_rate         = 0.0001
sft_trainer.hyperparameters.lora_rank             = 64
sft_trainer.hyperparameters.lora_alpha            = 256     # alpha/rank = 4
sft_trainer.hyperparameters.train_val_split_ratio = 0.9
sft_trainer.hyperparameters.lr_warmup_steps_ratio = 0.03
pprint(sft_trainer.hyperparameters.to_dict())

### Dry run, then train

`dry_run=True` validates permissions, datasets and hyperparameters without starting a job.

In [ ]:
sft_trainer.train(dry_run=True)
print("Dry run passed - configuration is valid.")

In [ ]:
training_job = sft_trainer.train(wait=False)
sft_trainer.stream_logs()                  # or: training_job = sft_trainer.train(wait=True)
print(training_job.training_job_name, training_job.training_job_status)

## 03. Get the Model Package

No merge step: the serverless job registers a ready-to-deploy Model Package (LoRA already merged).

In [ ]:
from sagemaker.core.resources import TrainingJob, ModelPackage

job = TrainingJob.get(training_job_name=training_job.training_job_name)
MODEL_PKG_ARN = job.output_model_package_arn
print("Output Model Package ARN:", MODEL_PKG_ARN, " <- set this in evaluate-tool-call-accuracy.ipynb")

model_package = ModelPackage.get(model_package_name=MODEL_PKG_ARN)
print("status:", model_package.model_package_status)

# To find the package later without the job object:
#   aws sagemaker list-model-packages --model-package-group-name seqkd-qwen35-4b-shopping \
#     --sort-by CreationTime --sort-order Descending \
#     --query 'ModelPackageSummaryList[0].ModelPackageArn'

## 04. Evaluate

Run `evaluate-tool-call-accuracy.ipynb` with the Model Package ARN from step 03. It measures **ASR** on held-out
ShoppingBench problems (does the student solve more tasks?) and, optionally, **IFEval** (does it keep general
instruction following?).